# NM i AI 2026 — Exercise 1: Object Detection
## Train YOLOv8m at 1280px with GPU

**Before running:** Upload `NM_NGD_coco_dataset.zip` (864 MB) to your Google Drive root folder.

**Runtime:** Change to **T4 GPU** → Runtime → Change runtime type → T4 GPU

## Step 1: Mount Drive & Extract Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust this path if you put the zip in a subfolder
ZIP_PATH = '/content/drive/MyDrive/NM_NGD_coco_dataset.zip'

!unzip -q {ZIP_PATH} -d /content/dataset
!ls /content/dataset/train/
!echo "Images:"
!ls /content/dataset/train/images/ | wc -l

## Step 2: Install ultralytics & Verify GPU

In [ ]:
!pip install "ultralytics>=8.3.0" -q

import torch
import torch.serialization as _ts
# PyTorch 2.6+: patch serialization.load + torch.load once (only patching torch.load can recurse on some versions).
if not getattr(_ts, "_ng_yolo_load_patched", False):
    _real_load = _ts.load
    def _ng_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _real_load(*args, **kwargs)
    _ts.load = _ng_load
    torch.load = _ng_load
    _ts._ng_yolo_load_patched = True

# Optional extra allowlist (safe_globals alone is often insufficient for nested pickles)
try:
    from ultralytics.nn.tasks import (
        DetectionModel,
        SegmentationModel,
        PoseModel,
        ClassificationModel,
    )
    if hasattr(torch.serialization, "add_safe_globals"):
        torch.serialization.add_safe_globals(
            [DetectionModel, SegmentationModel, PoseModel, ClassificationModel]
        )
except Exception as e:
    print("Note (safe_globals):", e)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3: Convert COCO → YOLO Format

In [ ]:
import json
import shutil
import random
from pathlib import Path
from collections import defaultdict

ann_path = Path('/content/dataset/train/annotations.json')
images_dir = Path('/content/dataset/train/images')
output_dir = Path('/content/yolo_data')

with open(ann_path) as f:
    data = json.load(f)

images = data['images']
annotations = data['annotations']
categories = data['categories']

img_id_to_info = {img['id']: img for img in images}
nc = len(categories)
print(f'Categories: {nc}, Images: {len(images)}, Annotations: {len(annotations)}')

anns_by_image = defaultdict(list)
for ann in annotations:
    anns_by_image[ann['image_id']].append(ann)

all_ids = sorted(img_id_to_info.keys())
random.seed(42)
random.shuffle(all_ids)
n_val = max(1, int(len(all_ids) * 0.2))
val_ids = set(all_ids[:n_val])
train_ids = set(all_ids[n_val:])

for split in ['train', 'val']:
    (output_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (output_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

skipped = 0
total = 0
for split_name, split_ids in [('train', train_ids), ('val', val_ids)]:
    for img_id in sorted(split_ids):
        info = img_id_to_info[img_id]
        fn, iw, ih = info['file_name'], info['width'], info['height']
        shutil.copy2(str(images_dir / fn), str(output_dir / 'images' / split_name / fn))
        lines = []
        for ann in anns_by_image.get(img_id, []):
            x, y, w, h = ann['bbox']
            if w <= 0 or h <= 0:
                skipped += 1
                continue
            xc = (x + w / 2) / iw
            yc = (y + h / 2) / ih
            lines.append(f"{ann['category_id']} {xc:.6f} {yc:.6f} {w/iw:.6f} {h/ih:.6f}")
            total += 1
        label_path = output_dir / 'labels' / split_name / f'{Path(fn).stem}.txt'
        label_path.write_text('\n'.join(lines) + ('\n' if lines else ''))

names_lines = '\n'.join(f"  {c['id']}: {c['name']}" for c in categories)
yaml_text = f"path: {output_dir}\ntrain: images/train\nval: images/val\nnc: {nc}\nnames:\n{names_lines}\n"
(output_dir / 'dataset.yaml').write_text(yaml_text)

print(f'Done! Train: {len(train_ids)}, Val: {len(val_ids)}')
print(f'Labels: {total}, Skipped: {skipped}')

## Step 4: Train YOLOv8m at 1280px
This takes ~30-40 minutes on T4 GPU.

In [ ]:
import torch
import torch.serialization as _ts
if not getattr(_ts, "_ng_yolo_load_patched", False):
    _real_load = _ts.load
    def _ng_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _real_load(*args, **kwargs)
    _ts.load = _ng_load
    torch.load = _ng_load
    _ts._ng_yolo_load_patched = True

import os
# Weights & Biases: Colab has wandb installed; it rejects Ultralytics save dir `project='/content/runs'` as wandb project name (no '/' allowed).
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data='/content/yolo_data/dataset.yaml',
    imgsz=1280,
    epochs=80,
    batch=8,
    patience=20,
    project='/content/runs',
    name='yolov8m_1280',
    exist_ok=True,
    close_mosaic=10,
    seed=42,
    workers=2,
    verbose=True,
    pretrained=True,
    val=True,
)

## Step 5: Check Results & Save to Drive

In [ ]:
import os
import shutil

best_pt = '/content/runs/yolov8m_1280/weights/best.pt'
last_pt = '/content/runs/yolov8m_1280/weights/last.pt'

if os.path.exists(best_pt):
    size_mb = os.path.getsize(best_pt) / (1024 * 1024)
    print(f'best.pt: {size_mb:.1f} MB')
else:
    print('ERROR: best.pt not found!')

# Copy to Google Drive
drive_dest = '/content/drive/MyDrive/best_yolov8m_1280.pt'
shutil.copy(best_pt, drive_dest)
print(f'\nSaved to Google Drive: {drive_dest}')
print('Download this file to your PC and place it in submission/best.pt')

## Step 6 (Optional): Quick Validation

In [ ]:
import torch
import torch.serialization as _ts
if not getattr(_ts, "_ng_yolo_load_patched", False):
    _real_load = _ts.load
    def _ng_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _real_load(*args, **kwargs)
    _ts.load = _ng_load
    torch.load = _ng_load
    _ts._ng_yolo_load_patched = True

import os
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

from ultralytics import YOLO

model_val = YOLO(best_pt)
metrics = model_val.val(
    data='/content/yolo_data/dataset.yaml',
    imgsz=1280,
    conf=0.10,
    iou=0.60,
    max_det=400,
)
print(f"\nmAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## Optional: Train YOLOv8l (larger model)
Only run this if yolov8m finished and you want to try the larger model.

In [ ]:
# model_l = YOLO('yolov8l.pt')
# results_l = model_l.train(
#     data='/content/yolo_data/dataset.yaml',
#     imgsz=1280,
#     epochs=80,
#     batch=4,
#     patience=20,
#     project='/content/runs',
#     name='yolov8l_1280',
#     exist_ok=True,
#     close_mosaic=10,
#     seed=42,
#     workers=2,
#     verbose=True,
#     pretrained=True,
# )
# shutil.copy('/content/runs/yolov8l_1280/weights/best.pt', '/content/drive/MyDrive/best_yolov8l_1280.pt')
# print('Saved yolov8l to Drive!')